<a href="https://colab.research.google.com/github/JSJeong-me/Retriever/blob/main/Multimodal_RAG/MRO_Excel_LangChain_Hybrid_RAG_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 기업 MRO 요구사항 처리 이력 — LangChain Hybrid Excel RAG

**대상:** `2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs(1).xls`

**목표:** 실제 MRO 요구사항 Excel에서 **① 정형 데이터 조건 검색·집계**와 **② 의미 기반 유사사례 검색**을 결합하고, 두 근거를 LangChain LLM에 전달하여 **근거 기반 답변**을 생성합니다.

> **실무 데이터 주의:** 회사명·담당자·고객 정보 등이 포함될 수 있습니다. 외부 API로 전송하기 전에 비식별화/보안 정책과 전송 권한을 확인하세요. 본 실습은 개인·고객 식별 정보의 외부 전송을 기본적으로 차단하도록 구성했습니다.

**학습 흐름:** xls 업로드 → 다중 헤더 점검 → 행 정규화 → 정형 필터/집계 → `Document` → OpenAI Embeddings + Chroma → Hybrid 검색 → ChatOpenAI → 근거·평가 검증.

## Cell 1. 라이브러리 설치

Colab에 필요한 패키지를 설치합니다. 구형 `.xls`는 `xlrd`를 통해 읽고, LangChain의 `ChatOpenAI`, `OpenAIEmbeddings`, `Chroma`를 사용합니다. **이 셀은 네트워크 연결이 있는 Colab에서 실행**하세요.

In [11]:
%pip -q install -U "xlrd>=2.0.1" pandas langchain-core langchain-openai langchain-chroma langchain-text-splitters chromadb ipython

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires ipython==7.34.0, but you have ipython 9.17.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.0 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.


## Cell 2. API 키와 실행 옵션

검색·집계 단계는 API 키 없이 수행할 수 있습니다. API 키는 벡터 임베딩을 만들거나 답변을 생성할 때만 필요합니다. `ENABLE_EXTERNAL_AI=False`이면 회사 원문을 외부 API로 보내지 않고 **정형 검색 데모만** 실행합니다.

임베딩/LLM에는 **비식별 요약 필드**만 전송하도록 설계했습니다. 단, 민감한 업무 내용이 있는 경우 요약문 자체도 검토하세요.

In [1]:
import os, re, json, hashlib, getpass, unicodedata
from pathlib import Path
import pandas as pd
from IPython.display import display

ENABLE_EXTERNAL_AI = True  # 기업 데이터의 외부 전송 승인을 받은 후에만 True로 변경
CHAT_MODEL = "gpt-4o-mini"
EMBED_MODEL = "text-embedding-3-small"
WORKDIR = Path("/content/mro_hybrid_rag") if Path("/content").exists() else Path.cwd() / "mro_hybrid_rag"
WORKDIR.mkdir(parents=True, exist_ok=True)

if ENABLE_EXTERNAL_AI and not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key: ")
print("외부 AI 사용:", ENABLE_EXTERNAL_AI, "| 작업 폴더:", WORKDIR)

OpenAI API Key: ··········
외부 AI 사용: True | 작업 폴더: /content/mro_hybrid_rag


## Cell 3. 원본 Excel 파일 업로드

첨부된 `.xls` 파일을 업로드합니다. Colab에서는 파일 선택창이 열립니다. 로컬 Jupyter에서 실행하는 경우 `EXCEL_PATH` 변수에 파일 경로를 지정하세요.

In [2]:
EXCEL_PATH = None  # 로컬 예: "/path/to/요구사항정의서.xls"
if EXCEL_PATH is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        assert uploaded, "업로드된 파일이 없습니다."
        EXCEL_PATH = next(iter(uploaded.keys()))
    except ImportError:
        matches = sorted(Path.cwd().glob("*.xls"))
        if not matches:
            raise FileNotFoundError("작업 디렉터리에 .xls가 없습니다. EXCEL_PATH를 지정하세요.")
        EXCEL_PATH = str(matches[0])
EXCEL_PATH = Path(EXCEL_PATH)
assert EXCEL_PATH.exists() and EXCEL_PATH.suffix.lower() == ".xls"
print("선택 파일:", EXCEL_PATH.name)

Saving 2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls to 2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls
선택 파일: 2026년_3분기_DO본부_MRO팀_접수일자기준-요구사항정의서(07월)_LineUs.xls


## Cell 4. 워크시트 및 실제 셀 구조 확인

업무용 요구사항 정의서에는 제목 행·병합 셀·다층 헤더가 있을 수 있습니다. **헤더를 임의로 첫 줄이라 가정하지 않고** 0~14행의 내용을 확인합니다. 아래 결과를 보고 `HEADER_ROW` 값을 조정하세요. `HEADER_ROW`는 **0부터 시작하는 행 번호**입니다.

In [3]:
excel = pd.ExcelFile(EXCEL_PATH, engine="xlrd")
print("시트 목록:", excel.sheet_names)
SHEET_NAME = "AS" if "AS" in excel.sheet_names else excel.sheet_names[0]
raw = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME, header=None, dtype=object, engine="xlrd").fillna("")
print("선택 시트:", SHEET_NAME, "| 원본 크기:", raw.shape)
for idx in range(min(15, len(raw))):
    vals = [f"{j}:{str(v)[:35]}" for j,v in enumerate(raw.iloc[idx]) if str(v).strip()][:15]
    print(f"행 {idx:02d}:", " / ".join(vals))

시트 목록: ['AS관리']
선택 시트: AS관리 | 원본 크기: (32, 56)
행 00: 0:접수정보 / 5:고객사정보 / 9:문의 유형 정보 / 16:처리상태 정보 / 22:처리완료 정보 / 34:처리완료 상세 정보 / 45:팀장승인 / 48:배포승인 / 51:배포정보 / 54:배포승인 정보
행 01: 0:No / 1:접수번호 / 2:하위작업 / 3:접수일 / 4:접수경로 / 5:거래처구분 / 6:거래처코드 / 7:거래처명 / 8:의뢰자명 / 9:처리상태 / 10:부서명 / 11:처리담당자 / 12:중요도 / 13:모듈 / 14:중분류
행 02: 0:30 / 1:20260731001 / 3:20260731 / 4:방문 / 5:본사 / 6:00000 / 7:(주)중외정보기술 / 8:김경아 / 9:배포승인 / 10:MRO팀 / 11:강롱기 / 12:일반건 / 13:ERP-인사 / 16:20260812 / 17:ERP
행 03: 0:29 / 1:20260729001 / 3:20260729 / 4:방문 / 5:본사 / 6:00000 / 7:(주)중외정보기술 / 8:이준호 / 9:처리완료 / 10:MRO팀 / 11:강롱기 / 12:일반건 / 16:20260729 / 17:ERP / 18:일반
행 04: 0:28 / 1:20260728004 / 3:20260728 / 4:CALL / 5:대외기업 / 6:11654 / 7:(주)한국티이아이 / 8:최정민 / 9:처리완료 / 10:MRO팀 / 11:최정민 / 12:일반건 / 16:20260729 / 17:기타 / 18:일반
행 05: 0:27 / 1:20260728003 / 3:20260728 / 4:기타 / 5:본사 / 6:00000 / 7:(주)중외정보기술 / 8:김경아 / 9:처리완료 / 10:MRO팀 / 11:김규민 / 12:일반건 / 16:20260724 / 17:A/S서비스플랫폼 / 18:일반
행 06: 0:26 / 1:20260727006 / 3:20260727 / 4:메신저 / 5:본사 / 6:00000 

## Cell 5. 다중 헤더 감지 및 컬럼 정규화

`접수번호`, `접수일자`, `처리상태` 등 핵심 업무 키워드가 집중된 행을 자동 제안합니다. 다층 헤더라면 `HEADER_DEPTH=2`로 변경할 수 있습니다. 중복 열명에는 접미 번호를 부여하며, 완전히 비어 있는 행·열을 제거합니다. **추천 행과 실제 출력 컬럼을 반드시 검수**하세요.

In [4]:
KEYWORDS = ["접수", "번호", "고객", "문의", "처리", "승인", "배포", "작업", "요청", "상태", "담당", "조치", "완료", "등록"]
def norm_header(v):
    return re.sub(r"\s+", " ", str(v).replace("\n", " ").strip())

def header_score(i):
    vals = [norm_header(x) for x in raw.iloc[i]]
    return sum(any(k in v for k in KEYWORDS) for v in vals if v)
scores = sorted(((header_score(i), i) for i in range(min(25, len(raw)))), reverse=True)
print("헤더 후보 (점수, 0부터 시작하는 행):", scores[:8])
HEADER_ROW = scores[0][1]  # 출력 확인 후 직접 수정할 수 있음
HEADER_DEPTH = 1            # 두 행에 걸친 헤더는 2로 설정

def build_frame(raw, header_row, header_depth):
    hdr = raw.iloc[header_row:header_row+header_depth].copy().replace("", pd.NA).ffill(axis=1).fillna("")
    names = []
    for col in range(raw.shape[1]):
        parts = [norm_header(x) for x in hdr.iloc[:,col].tolist()]
        parts = list(dict.fromkeys(p for p in parts if p))
        names.append(" / ".join(parts) or f"미지정_{col}")
    count = {}; unique = []
    for name in names:
        count[name] = count.get(name, 0)+1
        unique.append(name if count[name] == 1 else f"{name}__{count[name]}")
    out = raw.iloc[header_row+header_depth:].copy()
    out.columns = unique
    out.insert(0, "_excel_row", out.index + 1)  # 실제 Excel 행 번호(1-based)
    useful = [c for c in out.columns if c == "_excel_row" or not c.startswith("미지정_") or out[c].astype(str).str.strip().ne("").any()]
    out = out[useful]
    out = out.loc[out.drop(columns="_excel_row").astype(str).apply(lambda s:s.str.strip().ne("")).any(axis=1)].reset_index(drop=True)
    return out.fillna("")

df = build_frame(raw, HEADER_ROW, HEADER_DEPTH)
print(f"사용 헤더 행={HEADER_ROW}, 깊이={HEADER_DEPTH} | 추출 행={len(df)} / 컬럼={len(df.columns)}")
for i,c in enumerate(df.columns): print(f"{i:02d}: {c}")
display(df.head(3).T)

헤더 후보 (점수, 0부터 시작하는 행): [(29, 1), (10, 0), (6, 20), (6, 7), (5, 24), (5, 22), (5, 21), (5, 19)]
사용 헤더 행=1, 깊이=1 | 추출 행=30 / 컬럼=57
00: _excel_row
01: No
02: 접수번호
03: 하위작업
04: 접수일
05: 접수경로
06: 거래처구분
07: 거래처코드
08: 거래처명
09: 의뢰자명
10: 처리상태
11: 부서명
12: 처리담당자
13: 중요도
14: 모듈
15: 중분류
16: 프로그램명
17: 처리요청일자
18: 시스템유형
19: 업무유형
20: 문의유형
21: 요청내용
22: CMC담당자
23: 처리예정일자
24: 처리예정시각
25: 원인유형
26: 조치유형
27: 처리등급
28: 처리완료일
29: 예상 작업시간
30: 진행률
31: 대상 프로젝트
32: 작업시간
33: 조치 및 처리 의견
34: 프로그램 만족도
35: 처리구분
36: 빌드순번/CTS빌드
37: 파일명/PBL
38: 관련DB
39: 테스트케이스
40: 프로세스정의서
41: 기능분해도
42: 화면정의서
43: ERD
44: 테이블정의서
45: 인터페이스정의서
46: 승인여부
47: 승인자
48: 승인일자
49: 승인여부__2
50: 승인자__2
51: 승인일자__2
52: 배포경로 및 배포파일명
53: SVN버전
54: 배포일자
55: 배포승인일시
56: 배포승인시간


,0,1,2
_excel_row,3,4,5
No,30,29,28
접수번호,20260731001,20260729001,20260728004
하위작업,,,
접수일,20260731,20260729,20260728
접수경로,방문,방문,CALL
거래처구분,본사,본사,대외기업
거래처코드,00000,00000,11654
거래처명,(주)중외정보기술,(주)중외정보기술,(주)한국티이아이
의뢰자명,김경아,이준호,최정민


## Cell 6. 업무 의미에 따른 필드 매핑

원본 Excel의 실제 열 이름과 표기방식을 유지하면서, 분석에 필요한 필드를 **접수번호·상태·접수일·요청내용·조치의견·시스템·담당** 등 공통 키로 연결합니다. 자동 매핑이 잘못되었다면 `FIELD_OVERRIDE`에 실제 컬럼명을 직접 넣으세요. 검색 정확도에 매우 중요합니다.

In [5]:
ALIASES = {
 "ticket_id":["접수번호", "요구사항번호", "접수 no", "접수no"],
 "subtask":["하위작업", "세부작업", "작업순번"],
 "received_at":["접수일자", "접수일", "등록일자"],
 "status":["처리상태", "진행상태", "상태"],
 "system":["시스템", "시스템명", "구분", "서비스"],
 "category":["문의유형", "요청유형", "분류"],
 "title":["제목", "요구사항명", "요청제목", "문의제목"],
 "request":["요청내용", "요구사항", "문의내용", "접수내용", "요청사항", "내용"],
 "resolution":["조치내용", "조치의견", "처리내용", "처리결과", "해결내용", "완료내용"],
 "completed_at":["처리완료일", "완료일자", "완료일"],
 "work_hours":["작업시간", "소요시간", "투입시간"],
 "assignee":["처리담당자", "담당자", "담당"],
 "customer":["고객사명", "고객사", "고객"],
}
FIELD_OVERRIDE = {}  # 예: {"ticket_id":"접수정보 / 접수번호", "request":"문의내용"}
def flat(s): return re.sub(r"[^0-9a-zA-Z가-힣]", "", str(s).lower())
def match_col(key, choices):
    if key in FIELD_OVERRIDE:
        assert FIELD_OVERRIDE[key] in df.columns, f"열 없음: {FIELD_OVERRIDE[key]}"
        return FIELD_OVERRIDE[key]
    aliases = ALIASES[key]
    candidates = []
    for c in choices:
        cf = flat(c)
        for a in aliases:
            af = flat(a)
            if cf == af or cf.endswith(af) or af in cf:
                candidates.append((0 if cf == af else 1 if cf.endswith(af) else 2, len(cf), c)); break
    return sorted(candidates)[0][-1] if candidates else None
field_map = {k: match_col(k, df.columns) for k in ALIASES}
display(pd.DataFrame({"의미":list(field_map), "원본 컬럼":list(field_map.values())}))
assert field_map["ticket_id"] is not None, "접수번호 열을 지정하세요: FIELD_OVERRIDE['ticket_id']"
assert any(field_map[k] for k in ["request","title","resolution"]), "요청/조치 내용 컬럼을 수동 지정하세요."
print("전체 원본 행수:",len(df),"| 고유 접수번호:",df[field_map["ticket_id"]].astype(str).nunique())

,의미,원본 컬럼
0,ticket_id,접수번호
1,subtask,하위작업
2,received_at,접수일
3,status,처리상태
4,system,처리구분
5,category,문의유형
6,title,None
7,request,요청내용
8,resolution,None
9,completed_at,처리완료일


전체 원본 행수: 30 | 고유 접수번호: 27


## Cell 7. 요구사항 레코드 구성 및 민감정보 보호

Excel **한 행을 처리 이력 레코드**로 유지합니다. 접수번호가 빈 하위행은 (필요한 경우에만) 위쪽 값을 상속합니다. 고객명·담당자·개인 식별정보는 **LLM과 임베딩에 기본 전송하지 않습니다**. 이 정규화 단계가 검색과 집계에 사용되는 기준 데이터가 됩니다.

In [6]:
def clean_value(x):
    if pd.isna(x): return ""
    return re.sub(r"\s+", " ", str(x)).strip()

data = df.copy()
ticket_col = field_map["ticket_id"]
# 병합셀로 인해 접수번호가 아래 행에 공란으로 들어간 경우만 이어받음
raw_id = data[ticket_col].map(clean_value)
if raw_id.ne("").any(): data[ticket_col] = raw_id.replace("",pd.NA).ffill().fillna("")
records = []
for _,row in data.iterrows():
    record = {"_excel_row": int(row["_excel_row"])}
    for key,col in field_map.items(): record[key] = clean_value(row[col]) if col else ""
    record["ticket_id"] = re.sub(r"\.0$", "", record["ticket_id"])
    record["record_id"] = f"{SHEET_NAME}:{record['_excel_row']}"
    records.append(record)
records = [r for r in records if r['ticket_id'] or r['request'] or r['title']]
print("정규화 레코드:",len(records),"| 고유 접수번호:", len(set(r['ticket_id'] for r in records if r['ticket_id'])))
display(pd.DataFrame(records).drop(columns=["customer","assignee"],errors="ignore").head(5))

정규화 레코드: 30 | 고유 접수번호: 27


,_excel_row,ticket_id,subtask,received_at,status,system,category,title,request,resolution,completed_at,work_hours,record_id
0,3,20260731001,,20260731,배포승인,단순수정,프로그램 개선,,[프로그램] 평가 등록 화면 이동 로직 수정 요청 메인화면에서 출력되는 평개 갯수와...,,20260731,2,AS관리:3
1,4,20260729001,,20260729,처리완료,,데이터 수정,,[데이터] 발행 세금계산서 품목명 수정 요청 202607290002 20260729...,,20260729,0.5,AS관리:4
2,5,20260728004,,20260728,처리완료,,데이터 수정,,검사일자 수정,,20260729,0.5,AS관리:5
3,6,20260728003,,20260728,처리완료,,교육 진행,,병원용 라인어스 교육 진행 대상자 : 김범수 매니저,,20260724,0.5,AS관리:6
4,7,20260727006,,20260727,처리완료,,데이터 수정,,[데이터] 주문전표 삭제 요청 DO202607270003,,20260727,0.5,AS관리:7


## Cell 8. 정형 데이터 검색 — 접수번호·상태·시스템·키워드

정형 검색은 SQL의 `WHERE`와 같은 역할입니다. 접수번호는 정확하게 일치시키고, 상태와 시스템은 **원본 값으로 필터**하며, 날짜는 별도 전처리 없이 원본 필드에 나타난 문자열로 확인합니다. 없는 열을 가정해서 집계하지 않습니다.

In [7]:
def structured_search(ticket_id=None, status=None, system=None, keyword=None, limit=30):
    hits = []
    for r in records:
        if ticket_id and r['ticket_id'] != str(ticket_id): continue
        if status and str(status) not in r['status']: continue
        if system and str(system).lower() not in r['system'].lower(): continue
        searchable = " ".join(r.get(k,"") for k in ['title','request','resolution','category','status','system'])
        if keyword and str(keyword).lower() not in searchable.lower(): continue
        hits.append(r)
    return hits[:limit]

def show_hits(hits):
    cols = ['record_id','ticket_id','status','system','title','request','resolution']
    display(pd.DataFrame([{k:r.get(k,"")[:160] for k in cols} for r in hits]))

example_id = next((r['ticket_id'] for r in records if r['ticket_id']),"")
print("실제 데이터의 첫 접수번호 검색:",example_id)
show_hits(structured_search(ticket_id=example_id))

실제 데이터의 첫 접수번호 검색: 20260731001


,record_id,ticket_id,status,system,title,request,resolution
0,AS관리:3,20260731001,배포승인,단순수정,,[프로그램] 평가 등록 화면 이동 로직 수정 요청 메인화면에서 출력되는 평개 갯수와...,


## Cell 9. 정형 통계 — 상태·시스템별 건수와 처리시간

접수번호 하나에 하위작업 여러 행이 연결될 수 있어 **행 수와 접수 건수는 다릅니다**. 아래 셀은 둘 다 계산합니다. 처리시간은 값이 숫자로 안전하게 변환되는 경우에만 합산합니다. 원본 날짜 서식이 혼재하면 추가 정제가 필요합니다.

In [8]:
rframe = pd.DataFrame(records)
def group_metrics(group_key):
    if rframe.empty or not rframe[group_key].astype(str).str.strip().any():
        return pd.DataFrame(columns=[group_key,'행 수','고유 접수건'])
    sub = rframe.copy(); sub[group_key] = sub[group_key].replace("", "(미기재)")
    a = sub.groupby(group_key, dropna=False).agg(**{'행 수':('record_id','size'), '고유 접수건':('ticket_id',lambda x:x.replace('',pd.NA).nunique())})
    return a.sort_values('행 수',ascending=False).reset_index()
print("전체 행 수:",len(rframe),"| 고유 접수건:", rframe.ticket_id.replace('',pd.NA).nunique())
print("상태별 분포"); display(group_metrics('status'))
print("시스템별 분포"); display(group_metrics('system'))
if rframe.work_hours.astype(str).str.strip().any():
    nums = pd.to_numeric(rframe.work_hours.str.replace(',','',regex=False),errors='coerce')
    print('숫자로 해석 가능한 작업시간:',nums.notna().sum(),'건 / 합계:',nums.sum(),' (원본 시간 단위 확인 필수)')

전체 행 수: 30 | 고유 접수건: 27
상태별 분포


,status,행 수,고유 접수건
0,처리완료,22,22
1,배포승인,6,5
2,반려,1,1
3,처리중,1,1


시스템별 분포


,system,행 수,고유 접수건
0,(미기재),24,23
1,단순수정,5,4
2,형상변경,1,1


숫자로 해석 가능한 작업시간: 30 건 / 합계: 438.1  (원본 시간 단위 확인 필수)


## Cell 10. 의미검색용 LangChain Document 구축

요청/처리 본문을 하나의 문서로 만듭니다. `metadata`에는 `record_id`, `ticket_id`, `status`, `system`, `excel_row`를 저장합니다. **검색용 문서에는 고객사와 처리 담당자 정보를 넣지 않습니다.** 필요하면 현장 보안 규정을 만족하도록 추가 마스킹하세요.

In [9]:
from langchain_core.documents import Document

def redact(text):
    text = re.sub(r"[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}","[EMAIL]",str(text))
    text = re.sub(r"(?<!\d)01[016789][- ]?\d{3,4}[- ]?\d{4}(?!\d)","[PHONE]",text)
    text = re.sub(r"\b\d{6}[- ]?[1-4]\d{6}\b", "[PERSONAL_ID]",text)
    return text

def record_to_doc(r):
    body = '\n'.join([f"접수번호: {r['ticket_id']}",f"시스템: {r['system']}",f"유형: {r['category']}",f"상태: {r['status']}",f"제목: {r['title']}",f"요청내용: {r['request']}",f"조치내용: {r['resolution']}"])
    body = redact(body)
    return Document(page_content=body, metadata={"record_id":r['record_id'], "ticket_id":r['ticket_id'],"status":r['status'],"system":r['system'],"excel_row":r['_excel_row']})
documents = [record_to_doc(r) for r in records if r['request'] or r['title'] or r['resolution']]
print("문서 개수:",len(documents)); print(documents[0].page_content[:800] if documents else '문서 없음')

문서 개수: 30
접수번호: 20260731001
시스템: 단순수정
유형: 프로그램 개선
상태: 배포승인
제목: 
요청내용: [프로그램] 평가 등록 화면 이동 로직 수정 요청 메인화면에서 출력되는 평개 갯수와 메인화면에서 버튼클릭으로 들어왔을때 조회되는 리스트가 동일하게 수정 진행
조치내용: 


## Cell 11. Chroma 인덱스 구축 — 외부 전송 여부 선택

`ENABLE_EXTERNAL_AI=True`일 때만 문서의 비식별 텍스트를 OpenAI Embeddings로 전송하고 Chroma 벡터 인덱스를 생성합니다. 재실행 시 중복 문서가 쌓이지 않도록 메모리 컬렉션을 새로 만듭니다. 외부 전송이 승인되지 않으면 이 셀은 건너뛰는 것이 정상입니다.

In [12]:
vectorstore = None
if ENABLE_EXTERNAL_AI:
    assert os.environ.get("OPENAI_API_KEY"), "API 키가 필요합니다."
    from langchain_openai import OpenAIEmbeddings
    from langchain_chroma import Chroma
    if not documents: raise ValueError("인덱싱할 문서가 없습니다.")
    embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
    vectorstore = Chroma(collection_name="mro_training_hybrid", embedding_function=embeddings)
    vectorstore.add_documents(documents=documents, ids=[d.metadata['record_id'] for d in documents])
    print("Chroma 인덱싱 완료:",len(documents))
else:
    print("안전 모드: API 임베딩 생성 생략 (ENABLE_EXTERNAL_AI=False)")

Chroma 인덱싱 완료: 30


## Cell 12. Hybrid Retrieval: 정형 검색 + 의미 검색 합치기

한 질문에서 `filters`는 **필수 제약**으로 처리합니다. 의미 검색 문서도 조건에 맞는 것만 인정합니다. 접수번호가 주어지면 그 번호로 정형 검색을 먼저 수행하고, 해당 조건에 속하지 않는 유사 문서는 끼워 넣지 않습니다. 동시 점수 결합을 보여주는 기초 실습으로 정형 hit에 가중치를 부여합니다.

In [13]:
record_by_id = {r['record_id']:r for r in records}
def satisfies(r, filters):
    for k,v in (filters or {}).items():
        if v is None or v == '': continue
        if k == 'ticket_id' and r['ticket_id'] != str(v): return False
        if k in ['status','system'] and str(v).lower() not in r[k].lower(): return False
    return True

def hybrid_retrieve(question, filters=None, keyword=None, k=5):
    filters = filters or {}
    exact = [r for r in records if satisfies(r,filters) and (not keyword or keyword.lower() in ' '.join([r['title'],r['request'],r['resolution']]).lower())]
    # 모든 레코드를 무조건 결과로 만들지 않고, 조건이 없는 경우 키워드 매치만 정형 후보로 인정
    if not filters and not keyword: exact = []
    combined = {}
    for r in exact:
        combined[r['record_id']] = {"record":r,"score":2.0,"channels":["structured"]}
    if vectorstore is not None:
        for rank,doc in enumerate(vectorstore.similarity_search(question,k=min(12,max(k*3,8)))):
            rid = doc.metadata.get('record_id'); r=record_by_id.get(rid)
            if r is None or not satisfies(r,filters): continue
            if rid in combined:
                combined[rid]['score'] += 1.0/(rank+1)
                combined[rid]['channels'].append('semantic')
            else:
                combined[rid] = {'record':r,'score':1.0/(rank+1),'channels':['semantic']}
    return sorted(combined.values(), key=lambda h:-h['score'])[:k]

q = '메일 자동 발송 관련 요구사항과 처리 이력'
hits = hybrid_retrieve(q,keyword='메일',k=5)
for h in hits:
    r=h['record'];print(r['record_id'],r['ticket_id'],h['channels'],r['title'][:70])
print("참고: 안전 모드에서는 의미검색 채널이 실행되지 않습니다.")

AS관리:11 20260722002 ['structured', 'semantic'] 
AS관리:12 20260720002 ['semantic'] 
AS관리:5 20260728004 ['semantic'] 
AS관리:30 20260702004 ['semantic'] 
AS관리:10 20260722003 ['semantic'] 
참고: 안전 모드에서는 의미검색 채널이 실행되지 않습니다.


## Cell 13. 근거 기반 LangChain 답변 생성

검색 근거를 **접수번호 / Excel 행 번호**와 함께 LLM에 전달합니다. 요청과 조치가 기록되지 않은 부분은 추정하지 않습니다. 안전 모드에서는 **외부 호출 없이 검색 결과만 확인**합니다. 민감한 데이터 사용 승인을 받은 뒤에도 답변에 직접 식별정보가 포함되지 않도록 검토하세요.

In [14]:
def ask_mro(question, filters=None, keyword=None, k=5):
    hits = hybrid_retrieve(question,filters=filters,keyword=keyword,k=k)
    if not hits:
        return {'question':question,'answer':'지정한 조건에서 일치하는 근거를 찾지 못했습니다.','evidence':[]}
    evidence = []
    for h in hits:
        r=h['record']
        evidence.append({'record_id':r['record_id'],'ticket_id':r['ticket_id'],'excel_row':r['_excel_row'],'channels':h['channels'],'content':record_to_doc(r).page_content[:2200]})
    if not ENABLE_EXTERNAL_AI:
        return {'question':question,'answer':'안전 모드: LLM 호출 없이 검색 근거만 표시합니다.','evidence':evidence}
    from langchain_openai import ChatOpenAI
    from langchain_core.prompts import ChatPromptTemplate
    from langchain_core.output_parsers import StrOutputParser
    prompt = ChatPromptTemplate.from_messages([
        ('system','당신은 기업 MRO 업무요구사항 분석 담당자입니다. 오직 제공된 검색 근거만 사용하여 한국어로 대답하세요. 각 사실에 [접수번호 / Excel 행]을 표시하세요. 근거가 부족하면 부족하다고 답변하세요. 고객·개인 식별정보는 출력하지 마세요.'),
        ('human','질문: {question}\n\n검색 근거:\n{context}')])
    chain = prompt | ChatOpenAI(model=CHAT_MODEL,temperature=0) | StrOutputParser()
    ctx='\n\n'.join(f"[접수번호: {e['ticket_id']} / Excel 행: {e['excel_row']}]\n{e['content']}" for e in evidence)
    answer=chain.invoke({'question':question,'context':ctx})
    return {'question':question,'answer':answer,'evidence':evidence}

result = ask_mro('메일 자동 발송 관련 요구사항의 처리 이력을 정리해 주세요.',keyword='메일')
print(result['answer'])
for e in result['evidence']: print(f"- 근거: 접수번호 {e['ticket_id']}, Excel 행 {e['excel_row']}, 검색 채널 {e['channels']}")

메일 자동 발송 관련 요구사항의 처리 이력은 다음과 같습니다.

1. **접수번호: 20260722002 / Excel 행: 11**
   - 시스템: 형상변경
   - 유형: 프로그램 개선
   - 상태: 배포승인
   - 요청내용: AS별로 처리예정일 D-2 담당자에게 안내 메일 발송 로직 추가
   - 조치내용: 

이 외의 접수번호들은 메일 자동 발송과 관련된 내용이 아니므로 포함되지 않았습니다. 추가적인 요구사항이나 처리 이력이 필요하면 말씀해 주세요.
- 근거: 접수번호 20260722002, Excel 행 11, 검색 채널 ['structured', 'semantic']
- 근거: 접수번호 20260728004, Excel 행 5, 검색 채널 ['semantic']
- 근거: 접수번호 20260702004, Excel 행 30, 검색 채널 ['semantic']
- 근거: 접수번호 20260722003, Excel 행 10, 검색 채널 ['semantic']
- 근거: 접수번호 20260703001, Excel 행 29, 검색 채널 ['semantic']


## Cell 14. 기업 실습 — 접수번호 / 업무 분야 / 집계 질문

1. **정확 조회**: 실제 접수번호의 처리 이력을 찾습니다. 2. **유사사례**: 같은 업무 키워드에 해당하는 조치 유형을 탐색합니다. 3. **정형 통계**: 처리상태별 건수를 확인합니다. **통계 수치는 LLM이 아니라 DataFrame 집계 결과**를 사용합니다.

In [15]:
questions = [
    (f"접수번호 {example_id}의 요청과 처리 결과는 무엇입니까?", {'ticket_id':example_id},None),
    ('ERP 관련 개선 요구사항의 사례를 정리해 주세요.',None,'ERP'),
    ('자동 발송과 관련된 요구사항을 찾아주세요.',None,'자동'),
]
for question,flt,kw in questions:
    print('\n질문:',question)
    out = ask_mro(question,filters=flt,keyword=kw)
    print(out['answer']);print('검색 근거:',[(e['ticket_id'],e['excel_row']) for e in out['evidence']])
print('\n[정형 집계: 처리상태별 건수]');display(group_metrics('status'))


질문: 접수번호 20260731001의 요청과 처리 결과는 무엇입니까?
접수번호 20260731001의 요청은 "평가 등록 화면 이동 로직 수정 요청"으로, 메인화면에서 출력되는 평개 갯수와 메인화면에서 버튼 클릭으로 들어왔을 때 조회되는 리스트가 동일하게 수정하는 것입니다. 처리 결과는 "배포승인" 상태입니다. [접수번호: 20260731001 / Excel 행: 3]
검색 근거: [('20260731001', 3)]

질문: ERP 관련 개선 요구사항의 사례를 정리해 주세요.
ERP 관련 개선 요구사항의 사례는 다음과 같습니다:

1. **영림원 ERP 데이터 이관 과제 신규 개발**
   - 요청 내용:
     - 그룹웨어 법인마감일/전기마감일 추출 및 업무 분석
     - 적재 테이블 설계, 월/WBS 산출 구조 설계
     - 조회조건(맵핑파일) 설계 및 예외처리 설계
     - 영림원 ERP 화면 조회 및 데이터 다운로드 자동화 (JWM 9개, JWBS 29개)
     - 다운로드 데이터 검증 및 DB 적재 로직 개발
     - 기능 및 데이터 검증, 예외 케이스 테스트
     - 운영 환경 배포 및 초기 안정화
     - PDD, 운영가이드 등 개발 문서 작성
   - 접수번호: 20260707004 / Excel 행: 24, 22, 23

2. **ERP 권한 부여 요청**
   - 요청 내용:
     - 병원S&C팀 신규입사자 ERP 권한 부여 요청 (서동익 매니저님과 동일한 권한 요청)
   - 접수번호: 20260708001 / Excel 행: 21

3. **영림원 ERP RPA 과제 관련 회의 요청**
   - 요청 내용:
     - Data Lake 업체에서 영림원ERP RPA 과제에 대해 조회항목 논의 위한 회의 요청
   - 접수번호: 20260703003 / Excel 행: 27

이와 같은 사례들은 ERP 시스템의 데이터 이관, 권한 관리, 그리고 RPA 과제와 관련된 개선 요구사항을 포함하고 있습니다.
검색

,status,행 수,고유 접수건
0,처리완료,22,22
1,배포승인,6,5
2,반려,1,1
3,처리중,1,1


## Cell 15. 품질 평가 및 흔한 오류 점검

정확한 RAG의 기본은 **행/접수번호 출처 추적, 정형 필터 충족, 검색·응답 구분, 집계 단위 명시**입니다. 아래에서는 검색된 모든 근거가 지정한 접수번호 조건과 일치하는지 확인합니다. 다른 질문에 대해 `expected_ticket_id`를 교체하여 테스트하세요.

In [16]:
expected_ticket_id = example_id
check = ask_mro(f'{expected_ticket_id} 처리 현황',filters={'ticket_id':expected_ticket_id})
assert all(e['ticket_id']==expected_ticket_id for e in check['evidence']), '정형 필터 위반'
assert all(e['excel_row']>=1 for e in check['evidence']), '행 출처 누락'
assert len({r['record_id'] for r in records}) == len(records), '중복 record_id 발견'
print('필터 정확성, 행 출처, 레코드 고유성 점검 통과')
print('※ 헤더/필드 매핑 검수, 개인정보 점검, 검색 정답률, LLM 환각 검사는 별도 수동 평가가 필요합니다.')

필터 정확성, 행 출처, 레코드 고유성 점검 통과
※ 헤더/필드 매핑 검수, 개인정보 점검, 검색 정답률, LLM 환각 검사는 별도 수동 평가가 필요합니다.


## Cell 16. 확장 실습 제안

- **Hybrid 개선:** CSV/SQL 기반 메타데이터 필터, BM25 + Chroma 앙상블, Reciprocal Rank Fusion, 시간 범위 전처리
- **현업 KPI:** 요청 처리 리드타임(시작/완료 날짜 정의 확인), 미완료 backlog, 시스템별 이슈 분포, 유사 해결사례 재사용
- **보안:** 비식별 처리의 적정성 검토, 사내망 LLM/임베딩, 역할별 접근 통제, 로그 보존 기간
- **평가:** 접수번호 단위 Hit@k, 출처 일치율, 정형필터 준수율, 집계 결과 정확도, 근거 없는 답변 비율

**중요:** 이 데이터는 이미지 중심 PDF와 달리 구조화된 Excel이므로, 이미지 요약을 억지로 적용하기보다 **행별 의미 검색 + 정확 필터·집계**가 적합합니다. 통계 계산에는 RAG 답변을 쓰지 말고 원본 데이터를 사용하세요.

In [ ]:
import os
from datetime import datetime
from langchain_community.retrievers import BM25Retriever
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma

# 1. 날짜 범위 전처리 함수
def parse_date(date_str):
    try:
        return datetime.strptime(str(date_str).strip(), "%Y%m%d")
    except ValueError:
        try:
            return datetime.strptime(str(date_str).strip(), "%Y-%m-%d")
        except:
            return None

# 2. 메타데이터 필터 및 시간 범위 전처리 (SQL/CSV와 유사한 필터링)
def metadata_filter(records, start_date=None, end_date=None, system=None, status=None):
    filtered = []
    s_dt = datetime.strptime(start_date, "%Y-%m-%d") if start_date else None
    e_dt = datetime.strptime(end_date, "%Y-%m-%d") if end_date else None

    for r in records:
        # 날짜 필터링
        if s_dt or e_dt:
            rec_dt = parse_date(r.get('received_at', ''))
            if rec_dt:
                if s_dt and rec_dt < s_dt: continue
                if e_dt and rec_dt > e_dt: continue
            else:
                continue
        # 시스템 및 상태 필터링
        if system and system.lower() not in r.get('system', '').lower(): continue
        if status and status.lower() not in r.get('status', '').lower(): continue
        filtered.append(r)
    return filtered

# 3. Reciprocal Rank Fusion (RRF) 알고리즘
def reciprocal_rank_fusion(bm25_results, chroma_results, k=60):
    rrf_scores = {}

    def add_ranks(results):
        for rank, doc in enumerate(results):
            doc_id = doc.metadata["record_id"]
            if doc_id not in rrf_scores:
                rrf_scores[doc_id] = {"doc": doc, "score": 0.0}
            rrf_scores[doc_id]["score"] += 1.0 / (rank + k)

    add_ranks(bm25_results)
    add_ranks(chroma_results)

    # 스코어 기준 정렬
    sorted_docs = sorted(rrf_scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in sorted_docs]

# 4. 고급 앙상블 검색 & LLM 생성 함수
def advanced_hybrid_rag(question, start_date=None, end_date=None, system=None, status=None, top_n=5):
    # 1단계: 메타데이터 및 날짜 범위 필터링
    filtered_records = metadata_filter(records, start_date, end_date, system, status)
    if not filtered_records:
        return "조건에 맞는 레코드가 없습니다."

    filtered_docs = [record_to_doc(r) for r in filtered_records]

    # 2단계: 키워드 검색기 (BM25) 빌드
    bm25_retriever = BM25Retriever.from_documents(filtered_docs)
    bm25_retriever.k = min(len(filtered_docs), top_n * 2)
    bm25_hits = bm25_retriever.invoke(question)

    # 3단계: 의미론적 검색기 (Chroma) 빌드 (AI 사용 허가 시에만 동작)
    chroma_hits = []
    if ENABLE_EXTERNAL_AI and os.environ.get("OPENAI_API_KEY"):
        embeddings = OpenAIEmbeddings(model=EMBED_MODEL)
        # 메모리 내 Chroma에 조건 기반 문서 임시 로드
        temp_db = Chroma.from_documents(filtered_docs, embeddings)
        chroma_hits = temp_db.similarity_search(question, k=min(len(filtered_docs), top_n * 2))

    # 4단계: 앙상블 및 RRF 융합
    fused_docs = reciprocal_rank_fusion(bm25_hits, chroma_hits)[:top_n]

    # 5단계: LLM 호출을 통한 답변 생성
    context_text = "\n\n".join([
        f"[접수번호: {doc.metadata['ticket_id']} / Excel 행: {doc.metadata['excel_row']}]\n{doc.page_content}"
        for doc in fused_docs
    ])

    if not ENABLE_EXTERNAL_AI or not os.environ.get("OPENAI_API_KEY"):
        return {
            "status": "안전 모드 (LLM 호출 생략)",
            "fused_docs": [(doc.metadata['ticket_id'], doc.metadata['excel_row']) for doc in fused_docs],
            "context_preview": context_text[:500]
        }

    prompt = ChatPromptTemplate.from_messages([
        ('system', '당신은 숙련된 기업 MRO 담당자입니다. 제공된 검색 근거만을 바탕으로 한국어로 답변해 주세요. 대답할 때 [접수번호 / Excel 행] 형식을 포함하여 출처를 반드시 명시하세요.'),
        ('human', '질문: {question}\n\n검색 근거:\n{context}')
    ])

    chain = prompt | ChatOpenAI(model=CHAT_MODEL, temperature=0) | StrOutputParser()
    answer = chain.invoke({"question": question, "context": context_text})
    return {"answer": answer, "evidence": fused_docs}

# 실행 테스트 (7월 1일부터 7월 20일까지 접수된 단순수정 시스템 관련 문의에 대해 검색)
res = advanced_hybrid_rag(
    question="평가 등록 및 화면 이동 로직에 관한 이슈가 있나요?",
    start_date="2026-07-01",
    end_date="2026-07-31",
    system="단순수정"
)
import pprint
pprint.pprint(res)